# 12 - External Ontology Fetcher

Fetches external ontologies from namespace URIs and caches them to OneLake.

**Why a notebook?** Browser-based fetching is blocked by CORS policies. This notebook runs server-side where there are no CORS restrictions.

**Features:**
- Content negotiation (prefers Turtle, falls back to RDF/XML, JSON-LD)
- Follows redirects (common for ontology URIs)
- Validates response is RDF (not HTML)
- Caches to `Files/cache/external_ontologies/`

**Input:** JSON manifest at `Files/cache/fetch_manifest.json`  
**Output:** Cached ontology files + `Files/cache/fetch_results.json`

**Manifest Format:**
```json
{
  "uris": [
    "http://qudt.org/schema/qudt/",
    "https://w3id.org/def/basicsemantics-owl"
  ]
}
```

In [ ]:
# Configuration
LAKEHOUSE_PATH = "/lakehouse/default/Files"
CACHE_DIR = f"{LAKEHOUSE_PATH}/cache/external_ontologies"
MANIFEST_PATH = f"{LAKEHOUSE_PATH}/cache/fetch_manifest.json"
RESULTS_PATH = f"{LAKEHOUSE_PATH}/cache/fetch_results.json"

# HTTP settings
TIMEOUT_SECONDS = 30
USER_AGENT = "RDF2Fabric/1.0 (Ontology Fetcher)"

# Content negotiation - prefer Turtle
ACCEPT_HEADERS = ", ".join([
    "text/turtle",
    "application/rdf+xml",
    "application/n-triples",
    "application/ld+json",
    "text/n3",
    "*/*;q=0.1"
])

print(f"Cache directory: {CACHE_DIR}")
print(f"Manifest path: {MANIFEST_PATH}")

In [ ]:
import requests
import json
import os
import re
from urllib.parse import urlparse
from datetime import datetime
from typing import List, Dict, Optional, Tuple

# Content type to file extension mapping
CONTENT_TYPE_TO_EXT = {
    "text/turtle": ".ttl",
    "application/x-turtle": ".ttl",
    "text/n3": ".n3",
    "application/rdf+xml": ".rdf",
    "application/xml": ".rdf",
    "text/xml": ".rdf",
    "application/n-triples": ".nt",
    "application/ld+json": ".jsonld",
    "application/json": ".jsonld",
}

def uri_to_filename(uri: str) -> str:
    """Convert a URI to a safe filename."""
    try:
        parsed = urlparse(uri)
        domain = parsed.netloc.replace(".", "_")
        path_part = re.sub(r"[^a-zA-Z0-9]", "_", parsed.path)
        path_part = re.sub(r"_+", "_", path_part).strip("_")[:50]
        return f"{domain}_{path_part or 'index'}"
    except:
        return re.sub(r"[^a-zA-Z0-9]", "_", uri)[:60]

def get_extension_from_content_type(content_type: Optional[str]) -> str:
    """Detect file extension from content type header."""
    if not content_type:
        return ".ttl"
    mime_type = content_type.split(";")[0].strip().lower()
    return CONTENT_TYPE_TO_EXT.get(mime_type, ".ttl")

def looks_like_rdf(content: str) -> Tuple[bool, str]:
    """Check if content looks like RDF. Returns (is_rdf, reason)."""
    if not content or len(content) < 10:
        return False, "Empty or too short"
    
    # Check for HTML (common error response)
    if "<!DOCTYPE" in content or "<html" in content.lower():
        return False, "HTML response (server may not support content negotiation)"
    
    # Check for RDF patterns
    rdf_patterns = [
        "@prefix",        # Turtle
        "PREFIX",         # SPARQL-style Turtle
        "xmlns:",         # RDF/XML
        "<rdf:RDF",       # RDF/XML
        '"@context"',     # JSON-LD
        "<http",          # N-Triples
        "owl:",           # OWL namespace
        "rdfs:",          # RDFS namespace
    ]
    
    for pattern in rdf_patterns:
        if pattern in content:
            return True, f"Matched RDF pattern: {pattern}"
    
    return False, "No RDF patterns found"

print("Helper functions loaded.")

In [ ]:
def fetch_ontology(uri: str) -> Dict:
    """Fetch a single ontology from a URI."""
    result = {
        "uri": uri,
        "success": False,
        "cached_path": None,
        "format": None,
        "size_bytes": 0,
        "error": None,
        "timestamp": datetime.utcnow().isoformat()
    }
    
    try:
        print(f"  Fetching: {uri}")
        
        response = requests.get(
            uri,
            headers={
                "Accept": ACCEPT_HEADERS,
                "User-Agent": USER_AGENT,
            },
            timeout=TIMEOUT_SECONDS,
            allow_redirects=True
        )
        
        if response.status_code != 200:
            result["error"] = f"HTTP {response.status_code}: {response.reason}"
            print(f"    ✗ {result['error']}")
            return result
        
        content = response.text
        content_type = response.headers.get("Content-Type")
        
        # Validate it's RDF
        is_rdf, reason = looks_like_rdf(content)
        if not is_rdf:
            result["error"] = reason
            print(f"    ✗ {result['error']}")
            return result
        
        # Determine file extension and save
        ext = get_extension_from_content_type(content_type)
        filename = uri_to_filename(uri) + ext
        filepath = f"{CACHE_DIR}/{filename}"
        
        # Write to OneLake
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(content)
        
        result["success"] = True
        result["cached_path"] = f"cache/external_ontologies/{filename}"
        result["format"] = ext.replace(".", "")
        result["size_bytes"] = len(content.encode("utf-8"))
        
        print(f"    ✓ Cached: {filename} ({result['size_bytes']:,} bytes)")
        return result
        
    except requests.exceptions.Timeout:
        result["error"] = f"Timeout after {TIMEOUT_SECONDS}s"
    except requests.exceptions.ConnectionError as e:
        result["error"] = f"Connection error: {str(e)[:100]}"
    except Exception as e:
        result["error"] = f"Unexpected error: {str(e)[:100]}"
    
    print(f"    ✗ {result['error']}")
    return result

print("Fetch function loaded.")

In [ ]:
# Ensure cache directory exists
os.makedirs(CACHE_DIR, exist_ok=True)
print(f"Cache directory ready: {CACHE_DIR}")

# Load manifest
if not os.path.exists(MANIFEST_PATH):
    print(f"\n⚠ No manifest found at {MANIFEST_PATH}")
    print("Create a manifest with URIs to fetch, e.g.:")
    print("""{
  "uris": [
    "http://qudt.org/schema/qudt/",
    "https://w3id.org/def/basicsemantics-owl"
  ]
}""")
    uris_to_fetch = []
else:
    with open(MANIFEST_PATH, "r") as f:
        manifest = json.load(f)
    uris_to_fetch = manifest.get("uris", [])
    print(f"Loaded {len(uris_to_fetch)} URIs from manifest")

print(f"\nURIs to fetch: {len(uris_to_fetch)}")
for uri in uris_to_fetch:
    print(f"  - {uri}")

In [ ]:
# Fetch all ontologies
if not uris_to_fetch:
    print("No URIs to fetch. Add URIs to the manifest and re-run.")
    results = []
else:
    print(f"\n=== Fetching {len(uris_to_fetch)} ontologies ===")
    results = []
    
    for i, uri in enumerate(uris_to_fetch, 1):
        print(f"\n[{i}/{len(uris_to_fetch)}]")
        result = fetch_ontology(uri)
        results.append(result)
    
    # Summary
    success_count = sum(1 for r in results if r["success"])
    fail_count = len(results) - success_count
    total_bytes = sum(r["size_bytes"] for r in results)
    
    print(f"\n=== Summary ===")
    print(f"✓ Fetched: {success_count}")
    print(f"✗ Failed:  {fail_count}")
    print(f"Total:     {total_bytes:,} bytes")

In [ ]:
# Save results
if results:
    output = {
        "fetched_at": datetime.utcnow().isoformat(),
        "total": len(results),
        "success": sum(1 for r in results if r["success"]),
        "failed": sum(1 for r in results if not r["success"]),
        "results": results
    }
    
    with open(RESULTS_PATH, "w") as f:
        json.dump(output, f, indent=2)
    
    print(f"Results saved to: {RESULTS_PATH}")
    print(f"\nFailed URIs (if any):")
    for r in results:
        if not r["success"]:
            print(f"  {r['uri']}: {r['error']}")

In [ ]:
# Show cached files
print(f"\nCached ontology files in {CACHE_DIR}:")
if os.path.exists(CACHE_DIR):
    files = os.listdir(CACHE_DIR)
    for f in sorted(files):
        path = os.path.join(CACHE_DIR, f)
        size = os.path.getsize(path)
        print(f"  {f} ({size:,} bytes)")
    print(f"\nTotal: {len(files)} files")
else:
    print("  (no files yet)")